In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
import numpy as np

from testdata import mvn_with_correlation
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)

In [8]:
import numpy as np

def max_weighted_support_greedy(x, y, max_depth=5):
    n, p = x.shape
    orders = np.argsort(x, axis=0)
    support = np.ones(n, dtype=np.bool)
    support_count = n

    v = np.zeros(max_depth, dtype=np.int64)
    s = np.zeros(max_depth, dtype=np.int64)
    t = np.zeros(max_depth, dtype=np.float64)

    current = np.zeros(p, dtype=int) # cursor buffer for order updates
    
    best_sum = np.sum(y)
    num_cond = 0

    for k in range(1, max_depth+1):
        sum_y = np.sum(y[orders[:support_count, 0]])
        best_j, best_i, best_s = -1, -1, 1
        improvement = False
        for j in range(p):
            sum_left, sum_right = 0, sum_y
            for i in range(support_count - 1): 
                # test splits between x^j_i (last left) and x^j_i+1 (first right)
                y_i = y[orders[i, j]]
                sum_left += y_i
                sum_right -= y_i
                if sum_left > best_sum:
                    best_i = i
                    best_j = j
                    best_s = -1
                    best_sum = sum_left
                    improvement = True
                elif sum_right > best_sum:
                    best_i = i
                    best_j = j
                    best_s = 1
                    best_sum = sum_right
                    improvement = True

        if not improvement:
            break

        v[k-1] = best_j
        s[k-1] = best_s
        t[k-1] = (x[orders[best_i, best_j], best_j] + x[orders[best_i + 1, best_j], best_j]) / 2
        num_cond = k

        if best_s == 1: # lower bound
            support[orders[:best_i+1, best_j]] = False
            #orders[:support_count, best_j] = orders[best_i+1:best_i+1+support_count, best_j]
        else: # upper bound
            support[orders[best_i+1:, best_j]] = False

        current[:] = 0
        for i in range(support_count): # need old support count here
            for j in range(p): # can this loop be vectorised
                if support[orders[i, j]]:
                    orders[current[j], j] = orders[i, j]
                    current[j] += 1

        if best_s == 1: # lower bound
            support_count = support_count - best_i - 1
        else: # upper bound
            support_count = best_i + 1
    
    return v[:num_cond], s[:num_cond], t[:num_cond], best_sum

max_weighted_support_greedy(x, y)

(array([3, 0, 1]),
 array([-1, -1,  1]),
 array([ 0.49419261,  1.51353629, -0.14016027]),
 np.float64(21.443390305350473))

In [3]:
from optikon import max_weighted_support, equal_width_propositionalization

max_weighted_support(x, y, equal_width_propositionalization(x))

(array([ 8, 16, 59, 66]), 20.105766513335315, 5244, 22996)

In [10]:
from testdata import SMALL_1

max_weighted_support_greedy(SMALL_1.x, SMALL_1.y)


(array([0, 2]), array([1, 1]), array([0.5, 0.5]), np.int64(3))